In [ ]:
import asyncio, json, openai
from tqdm.asyncio import tqdm as tqdm_async
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
)
from pathlib import Path
from collections import Counter
import pandas as pd
import os

### Helper Funcs and Constants

In [ ]:
DATA_DIR = Path("../data")

ENV_PATH = Path("/Users/navneetmann/Documents/Code/MSc_Final_Thesis/MSc_Thesis/.env")
INPUT_CSV = DATA_DIR / "combined_2017_2023_cleaned.csv"
OUTPUT_CSV = DATA_DIR / "combined_2017_2023_themes_with_KG.csv"
RESULTS_PATH = Path("gold_std_results.json")
CHECKPOINT_PATH = Path("checkpoint.json")
FAILURES_PATH = Path("failures.json")

MAX_TOKENS = 8192
MODEL_NAME = "deepseek-v4-pro"
BASE_URL = "https://api.deepseek.com"
CONCURRENCY = 100

BATCH_SIZE = 500
N_SAMPLES = 3000
THEME_QUOTA = 100
RANDOM_STATE = 72

ENTITY_TYPES = {
    "ORG",
    "ORG/GOV",
    "ORG/REG",
    "PERSON",
    "GPE",
    "COMP",
    "PRODUCT",
    "EVENT",
    "SECTOR",
    "ECON_INDICATOR",
    "FIN_INSTRUMENT",
    "CONCEPT",
}
THEME_RELATIONS = {"HAS_ACTOR", "HAS_TARGET", "HAS_CONTEXT", "AFFECTS"}
ENTITY_RELATIONS = {"ACTS_ON", "BELONGS_TO"}
ALL_RELATIONS = THEME_RELATIONS | ENTITY_RELATIONS
DIMENSIONS = {
    "economic_monetary_event",
    "geopolitical_factor",
    "sector_or_industry",
    "policy_or_regulation",
    "technology_concept",
    "macro_trend",
}

In [ ]:
client = openai.AsyncOpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url=BASE_URL)
semaphore = asyncio.Semaphore(CONCURRENCY)

In [ ]:
THEME_KEYWORDS = {
    "AI": ["llm", "gpt", "neural", "machine learning", "generative", "chatgpt"],
    "Covid19 pandemic": ["covid", "pandemic", "coronavirus", "lockdown", "vaccine"],
    "USA Tariff": ["tariff", "trade war", "import duty", "protectionist", "customs"],
    "Russia - Ukraine war": [
        "ukraine",
        "russia",
        "nato",
        "zelensky",
        "kremlin",
        "sanctions",
    ],
    "Oil Crisis": ["opec", "crude", "brent", "barrel", "energy crisis"],
    "Semiconductor Chips": ["tsmc", "chip", "semiconductor", "wafer", "silicon"],
    "Fed interest rates": [
        "federal reserve",
        "fomc",
        "rate hike",
        "rate cut",
        "basis points",
    ],
}

In [ ]:
SYSTEM_PROMPT = """You are an expert financial news analyst and knowledge graph extractor.
 
Given the news article below, produce a single JSON object with exactly 
two keys: "predefined_themes" and "knowledge_graph".
 
Process in this strict order:
  Step 1 → Evaluate predefined_themes
  Step 2 → Identify themes          (working step — not output)
  Step 3 → Build knowledge_graph anchored to those themes
 
═══════════════════════════════════════════════════
STEP 1 — PREDEFINED THEMES
═══════════════════════════════════════════════════
For each theme below, return "Yes" if clearly present in the article, 
else return "NA":
  - AI
  - Covid19 pandemic
  - USA Tariff
  - Russia - Ukraine war
  - Oil Crisis
  - Semiconductor Chips
  - Fed interest rates
 
═══════════════════════════════════════════════════
STEP 2 — IDENTIFY THEMES
═══════════════════════════════════════════════════
A theme is a TOPIC TAG — the kind of label a financial analyst would file 
the article under. It is not a fact, and it is not a name.
 
Two tests. A phrase must pass both:
  TEST 1 — Could this same phrase tag a different article, six months from 
           now, about different companies? If no, it is a fact. Reject it.
  TEST 2 — Does it contain a company, person, place, product, or number? 
           If yes, it is an entity. Reject it.
 
Shape: 2–5 words. A noun phrase. Understandable on its own.
 
Go through the 6 dimensions below. For each one, write ONE theme phrase if 
that dimension is present in the article — otherwise skip it. Do not force 
a phrase. Most articles will yield 2–3 themes, not 6.
 
  DIMENSION                  GOOD                            BAD
  ─────────────────────────  ──────────────────────────────  ─────────────────────────
  economic_monetary_event    "central bank rate tightening"  "Fed hiked 50bps"
  geopolitical_factor        "cross border trade tension"    "Russia invaded Ukraine"
  sector_or_industry         "semiconductor supply shortage" "TSMC fab delay"
  policy_or_regulation       "import tariff escalation"      "Section 301 tariffs"
  technology_concept         "large language model adoption" "GPT-4 launch"
  macro_trend                "persistent inflation pressure" "CPI rose 3.2%"
 
The BAD column fails because each one names a specific company, person, 
number, or one-off occurrence. The GOOD column would still make sense as a 
tag on an article you have never seen.
 
Write each theme phrase down once, exactly as you will use it. You will 
repeat it verbatim in every triplet in Step 3 — same words, same order, 
same spelling, every time. Do not shorten or reword it later.
 
═══════════════════════════════════════════════════
STEP 3 — KNOWLEDGE GRAPH
═══════════════════════════════════════════════════
The knowledge graph exists to describe the themes from Step 2 — nothing else.
Every triplet is built AROUND one of those themes and must trace back to it,
directly or in one hop. If a fact in the article does not connect to any
discovered theme, leave it out. No free-floating facts; no theme without at
least one triplet.
 
Take the themes from Step 2 one at a time. For each theme:
 
  a) The theme phrase is a node of type CONCEPT
  b) Link the theme to real-world entities from the article using the 
     theme-to-entity relationships below
  c) Expand outward: add entity-to-entity triplets that explain why that 
     theme matters in the article
  d) Drop any triplet that cannot be connected — directly or in one hop — 
     back to a theme
  e) Tag every triplet with its owning theme phrase and dimension
 
Finish one theme completely before starting the next.
 
RELATIONSHIPS — use only these:
 
  Theme-to-entity (the theme is always the HEAD):
    HAS_ACTOR     → who initiates or drives the theme
    HAS_TARGET    → what the theme is directed at
    HAS_CONTEXT   → the sector, region, indicator, instrument, or product 
                    the theme operates in
    AFFECTS       → who or what the theme impacts
 
  Entity-to-entity (support, one hop from the theme):
    ACTS_ON       → one entity does something to another (announces, 
                    introduces, controls, invests in, impacts)
    BELONGS_TO    → membership, ownership, sector placement, or location
 
ENTITY TYPES — use exactly one per entity:
  ORG           → Organizations (non-government, non-regulatory)
  ORG/GOV       → Government bodies
  ORG/REG       → Regulatory bodies
  PERSON        → Individuals
  GPE           → Countries, cities, geopolitical entities
  COMP          → Companies
  PRODUCT       → Products or services
  EVENT         → Specific material events
  SECTOR        → Industries or company sectors
  ECON_INDICATOR → Economic indicators (not raw numbers/percentages)
  FIN_INSTRUMENT → Financial instruments, markets
  CONCEPT       → Theme anchor nodes ONLY — never create a CONCEPT node 
                  that is not a theme phrase from Step 2
 
CONSTRAINTS:
  • No generic, numerical, or temporal entities
  • No redundant triplets
  • No strictly past-tense events
  • Disambiguate entities (e.g. "BOE" → "Bank of England")
  • Max 4 words per entity label
  • Every triplet must belong to exactly one discovered theme — either a
    theme-to-entity edge, or an entity-to-entity edge one hop from that theme
  • The ONLY CONCEPT nodes allowed are the Step 2 theme phrases; never invent
    an extra CONCEPT node (e.g. do not link one theme to another new concept)
  • Give every discovered theme at least one triplet
 
═══════════════════════════════════════════════════
OUTPUT FORMAT — return ONLY valid JSON, no extra text
═══════════════════════════════════════════════════
Do not output the Step 2 theme list as its own key. Themes appear only in 
the "theme" field of each triplet.
 
{
  "predefined_themes": {
    "AI": "Yes | NA",
    "Covid19 pandemic": "Yes | NA",
    "USA Tariff": "Yes | NA",
    "Russia - Ukraine war": "Yes | NA",
    "Oil Crisis": "Yes | NA",
    "Semiconductor Chips": "Yes | NA",
    "Fed interest rates": "Yes | NA"
  },
  "knowledge_graph": [
    {
      "triplet": ["head_entity", "head_type", "relationship", "tail_entity", "tail_type"],
      "theme": "the Step 2 theme phrase — identical in every triplet that belongs to it",
      "dimension": "economic_monetary_event | geopolitical_factor | sector_or_industry | policy_or_regulation | technology_concept | macro_trend"
    }
  ]
}

═══════════════════════════════════════════════════
INPUT_TEXT:
"""

In [ ]:
@retry(
    retry=retry_if_exception_type((openai.RateLimitError, openai.APIError, ValueError)),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    stop=stop_after_attempt(3),
)
async def call_LLM(prompt, article_id):

    async with semaphore:
        response = await client.chat.completions.create(
            model=MODEL_NAME,
            max_tokens=MAX_TOKENS,
            response_format={"type": "json_object"},
            extra_body={"thinking": {"type": "disabled"}},
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
        )

        content = response.choices[0].message.content
        if not content:
            raise ValueError(f"Empty response for article {article_id}")

        return {"article_id": article_id, "result": json.loads(content)}


async def batch_process(
    prompts,
    article_ids,
    batch_size=BATCH_SIZE,
    checkpoint_path=CHECKPOINT_PATH,
    failures_path=FAILURES_PATH,
):

    # pick up from checkpoint.json
    if checkpoint_path.exists():
        all_results = json.loads(checkpoint_path.read_text())
        done_ids = {r["article_id"] for r in all_results}
        print(f"Resuming from checkpoint: {len(done_ids)} articles already done")

    else:
        all_results = []
        done_ids = set()

    all_failures = []

    # filter for not processed articles
    remaining = [
        (article_id, p)
        for article_id, p in zip(article_ids, prompts)
        if article_id not in done_ids
    ]

    # split articles into batches (for concurrency)
    article_batches = []
    for start in range(0, len(remaining), batch_size):
        article_batches.append(remaining[start : start + batch_size])

    with tqdm_async(total=len(remaining), desc="KG extraction", unit="doc") as pbar:

        for article_batch in article_batches:
            tasks = [call_LLM(p, aid) for aid, p in article_batch]
            results = await asyncio.gather(*tasks, return_exceptions=True)

            # loop through results and collect failures
            for (article_id, _), result in zip(article_batch, results):

                if isinstance(result, Exception):
                    all_failures.append(
                        {
                            "article_id": article_id,
                            "error": f"{type(result).__name__}: {result}",
                        }
                    )

                else:
                    all_results.append(result)

                pbar.update(1)

            # save after every batch (fault tolerance)
            checkpoint_path.write_text(json.dumps(all_results, indent=2))

    # debug failures
    if all_failures:
        failures_path.write_text(json.dumps(all_failures, indent=2))
        print(f"Saved {len(all_failures)} failures to {failures_path}")

    print(f"Done: {len(all_results)} succeeded, {len(all_failures)} failed")

    return all_results

In [ ]:
def stratified_sample(df, n, year_column="year"):
    """sample n rows, keeping each year's share of the data roughly the same"""

    counts = df[year_column].value_counts()
    fracs = counts / counts.sum()

    # stratified sample on basis of year
    sampled = df.groupby(year_column, group_keys=False).apply(
        lambda g: g.sample(
            n=max(1, round(fracs[g.name] * n)),
            random_state=RANDOM_STATE,
        ),
        include_groups=False,
    )

    # shuffle and return
    return (
        sampled.sample(frac=1, random_state=RANDOM_STATE).head(n).reset_index(drop=True)
    )


def theme_aware_sample(
    df,
    n,
    theme_quota=1,
    year_column="year",
    text_cols=["description", "maintext"],
):

    # lowercase text
    article_text = df[text_cols].fillna("").apply(" ".join, axis=1).str.lower()

    selected_ids = set()
    theme_rows = []

    # grab articles from each theme
    for theme, keywords in THEME_KEYWORDS.items():

        # simple regex to catch on theme keywords
        mask = article_text.str.contains("|".join(keywords), regex=True)
        candidates = df[mask & ~df["article_id"].isin(selected_ids)]

        if not candidates.empty:
            chosen = candidates.sample(
                n=min(theme_quota, len(candidates)), random_state=RANDOM_STATE
            )
            theme_rows.append(chosen)
            selected_ids.update(chosen["article_id"])
            print(f"{theme}: picked {len(chosen)} article(s)")

        else:
            print(f"{theme}: no candidates found")

    if theme_rows:
        theme_df = pd.concat(theme_rows).drop_duplicates("article_id")
    else:
        theme_df = pd.DataFrame()

    # fill the rest with a general stratified sampled articles
    remaining_n = n - len(theme_df)
    if remaining_n > 0:
        pool = df[~df["article_id"].isin(selected_ids)]
        generic_df = stratified_sample(pool, remaining_n, year_column)
    else:

        generic_df = pd.DataFrame()

    # combine and sample the final result
    result = (
        pd.concat([theme_df, generic_df])
        .drop_duplicates("article_id")
        .sample(frac=1, random_state=RANDOM_STATE)
        .head(n)
        .reset_index(drop=True)
    )

    print(
        f"Sample: {len(theme_df)} theme specific + {len(generic_df)} general = {len(result)} total articles"
    )

    return result

### Call API

In [ ]:
# this has initial 70k articles
combined = pd.read_csv(INPUT_CSV)

In [ ]:
# skip articles we have already processed
processed_ids = set()
for path in [RESULTS_PATH, CHECKPOINT_PATH]:
    if path.exists():
        processed_ids.update(r["article_id"] for r in json.loads(path.read_text()))


unprocessed = combined[~combined["article_id"].isin(processed_ids)]
print(
    f"Skipping {len(processed_ids)} already-processed articles, {len(unprocessed)} remain"
)

# sample articles (favour topics in THEME_KEYWORDS)
sample_df = theme_aware_sample(unprocessed, N_SAMPLES, theme_quota=THEME_QUOTA)

# combine description + main text into one
sample_df["news_text"] = (
    sample_df["description"].fillna("").str.strip()
    + "\n\n"
    + sample_df["maintext"].fillna("").str.strip()
).str.strip()

print(f"Total rows: {len(sample_df)}")

In [ ]:
article_ids = sample_df["article_id"].tolist()
prompts = sample_df["news_text"].tolist()
gold_std_results = await batch_process(prompts, article_ids)

In [ ]:
out_path = RESULTS_PATH

# load prev results so new runs dont overwrite
if out_path.exists():
    existing = json.loads(out_path.read_text())
    existing_ids = {r["article_id"] for r in existing}
    new_results = [r for r in gold_std_results if r["article_id"] not in existing_ids]
    merged = existing + new_results

else:
    existing = []
    merged = gold_std_results

out_path.write_text(json.dumps(merged, indent=2))
print(
    f"Saved {len(merged)} total results to {out_path} ({len(merged) - len(existing)} new)"
)

In [ ]:
kg_raw = json.loads(RESULTS_PATH.read_text())

# look up a model result by article id
results_by_id = {r["article_id"]: r["result"] for r in kg_raw}

# add the model output as new columns
combined_kg = combined.copy()

predefined_col = []
kg_col = []
for aid in combined_kg["article_id"]:
    if aid in results_by_id:
        predefined_col.append(json.dumps(results_by_id[aid]["predefined_themes"]))
        kg_col.append(json.dumps(results_by_id[aid]["knowledge_graph"]))
    else:
        predefined_col.append(None)
        kg_col.append(None)

combined_kg["predefined_themes"] = predefined_col
combined_kg["knowledge_graph"] = kg_col

print(f"Total rows: {len(combined_kg)}")
print(f"Rows with KG data: {combined_kg['predefined_themes'].notna().sum()}")

In [ ]:
out_csv = OUTPUT_CSV
combined_kg.to_csv(out_csv, index=False)
print(f"Saved {len(combined_kg)} rows to {out_csv}")

### Clean up malformed KGs

In [ ]:
# filter only processed articles with KG
df = combined_kg[~combined_kg["predefined_themes"].isnull()]
df.head()

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(df[["maintext", "predefined_themes", "knowledge_graph"]].iloc[0].to_frame())

In [ ]:
def clean_kg(kg, relax_concepts=False):

    cleaned = []
    seen = set()
    drops = Counter()

    # collect every theme phrase in article's KG
    article_themes = {
        e["theme"].strip().lower()
        for e in kg
        if isinstance(e, dict)
        and isinstance(e.get("theme"), str)
        and e["theme"].strip()
    }

    for entry in kg:

        if not isinstance(entry, dict):
            drops["not_dict"] += 1
            continue

        triplet = entry.get("triplet")
        theme = entry.get("theme")
        dimension = entry.get("dimension")

        # check basic shape
        if not isinstance(triplet, (list, tuple)) or len(triplet) != 5:
            drops["malformed_triplet"] += 1
            continue

        if not all(isinstance(x, str) and x.strip() for x in triplet):
            drops["empty_triplet"] += 1
            continue

        if not (isinstance(theme, str) and theme.strip()):
            drops["empty_theme"] += 1
            continue

        # extract triplet elements
        head, htype, rel, tail, ttype = (x.strip() for x in triplet)
        htype, ttype, rel = htype.upper(), ttype.upper(), rel.upper()
        dimension = str(dimension).strip().lower()
        theme = theme.strip()

        # check relation and dimension against the ontology
        if htype not in ENTITY_TYPES:
            drops["invalid_head_type"] += 1
            continue

        if ttype not in ENTITY_TYPES:
            drops["invalid_tail_type"] += 1
            continue

        if rel not in ALL_RELATIONS:
            drops["invalid_relation"] += 1
            continue

        if dimension not in DIMENSIONS:
            drops["invalid_dimension"] += 1
            continue

        # CONCEPT nodes are only allowed to be theme anchors
        # strict -> the row's own theme; relaxed -> any theme in this article
        valid_concepts = article_themes if relax_concepts else {theme.lower()}
        # a theme-to-entity relation must touch the theme's CONCEPT node
        if rel in THEME_RELATIONS and "CONCEPT" not in (htype, ttype):
            drops["theme_rel_no_concept"] += 1
            continue
        if htype == "CONCEPT" and head.lower() not in valid_concepts:
            drops["concept_not_theme"] += 1
            continue
        if ttype == "CONCEPT" and tail.lower() not in valid_concepts:
            drops["concept_not_theme"] += 1
            continue

        # drop self-loops and exact duplicates
        if head.lower() == tail.lower():
            drops["self_loop"] += 1
            continue
        key = (head.lower(), htype, rel, tail.lower(), ttype, theme.lower())
        if key in seen:
            drops["duplicate_triplet"] += 1
            continue

        seen.add(key)

        cleaned.append(
            {
                "triplet": [head, htype, rel, tail, ttype],
                "theme": theme,
                "dimension": dimension,
            }
        )

    return cleaned, drops


def extract_themes(kg):
    """unique (theme, dimension) pairs, keeping the order they first appear"""

    themes = []
    seen = set()
    for t in kg:
        pair = (t["theme"], t["dimension"])

        if pair not in seen:
            seen.add(pair)
            themes.append({"theme": t["theme"], "dimension": t["dimension"]})

    return themes

In [ ]:
# Clean every article's KG against the ontology and split the themes into a new column.
total_drops = Counter()
n_raw = 0
n_clean = 0
kg_clean_col = []
themes_col = []

for aid in combined_kg["article_id"]:

    result = results_by_id.get(aid)

    if result is None:
        kg_clean_col.append(None)
        themes_col.append(None)
        continue

    kg = result.get("knowledge_graph", [])
    cleaned, drops = clean_kg(kg, relax_concepts=True)
    n_raw += len(kg)
    n_clean += len(cleaned)
    total_drops.update(drops)
    kg_clean_col.append(json.dumps(cleaned))
    themes_col.append(json.dumps(extract_themes(cleaned)))

combined_kg["knowledge_graph"] = kg_clean_col
combined_kg["themes"] = themes_col

# report how many triplets we kept and why the rest were dropped
n_dropped = n_raw - n_clean
print(
    f"Triplets: {n_raw:,} raw, {n_clean:,} kept, {n_dropped:,} dropped ({n_dropped / max(n_raw, 1):.1%})\n"
)
print("Dropped by reason:")
for reason, count in total_drops.most_common():
    print(f"{reason}: {count:,}")

# save the cleaned KG + themes column (replaces the raw save above)
combined_kg.to_csv(OUTPUT_CSV, index=False)
print(f"Saved cleaned KG + themes to {OUTPUT_CSV}")